In [1]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils import resample
from agents.ffnn_agent2 import FFNNAgent
from torch.utils.data import TensorDataset, DataLoader
import torch
import matplotlib.pyplot as pltimage
import random
import os
import json
from itertools import cycle
from typing import List, Dict

In [3]:
import re
from pathlib import Path

# ---------- Helpers ----------

def is_seed_dir(p: Path) -> bool:
    return (
        p.is_dir()
        and p.name.startswith("seed_")
        and (p / "metrics.csv").is_file()
        and (p / "meta.json").is_file()
    )

def is_experiment_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    return any(is_seed_dir(d) for d in p.iterdir())

# Regex to extract PCA, minID, majID, BIAS
CORE_RE = re.compile(
    r"PCA(?P<pca>\d+).*?minID(?P<min>\d+).*?majID(?P<maj>\d+).*?BIAS(?P<bias>[0-9.]+)",
    re.IGNORECASE,
)

def make_short_label(exp_name: str) -> str:
    """
    Extract PCA, minID, majID, bias from a folder name.
    Return a short readable label like:
        PCA4_min1_maj0_bias0.35
    """
    m = CORE_RE.search(exp_name)
    if not m:
        # fallback: return the core part without DATE
        core = exp_name.split("_DATE")[0]
        return core

    pca = m.group("pca")
    min_id = m.group("min")
    maj_id = m.group("maj")
    bias = m.group("bias")

    return f"PCA{pca}_min{min_id}_maj{maj_id}_bias{bias}"

# ---------- Main Generator ----------

def generate_experiment_labels_dict():
    training_root = Path("training_runs")
    if not training_root.is_dir():
        print("[error] training_runs/ does not exist.")
        return

    exp_dirs = sorted(
        [d for d in training_root.iterdir() if is_experiment_dir(d)],
        key=lambda p: p.name,
    )

    if not exp_dirs:
        print("[info] No experiment folders found.")
        return

    print("\n# ================= GENERATED EXPERIMENT_LABELS =================")
    print("EXPERIMENT_LABELS = {")
    for d in exp_dirs:
        short = make_short_label(d.name)
        print(f'    "{d.name}": "{short}",')
    print("}")
    print("# ==============================================================\n")

# Run immediately
generate_experiment_labels_dict()



# ================= GENERATED EXPERIMENT_LABELS =================
EXPERIMENT_LABELS = {
    "SPECablation2_credit_global_diversity_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202602251537_06a4e33b": "SPECablation2_credit_global_diversity_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202602251537_06a4e33b",
    "SPECablation2_credit_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071731_c2324064": "SPECablation2_credit_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071731_c2324064",
    "SPECablation2_credit_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071741_9987eee9": "SPECablation2_credit_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071741_9987eee9",
    "SPECablation2_credit_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202602251532_f47867a4": "SPECablation2_credit_global_only_EP6000_PCA10_REWfairness_minID1_majID0_

In [ ]:
"""
Drop-in plotting script for training_runs/

What it does:
1) Auto-discovers ALL numeric time-series metrics in seed_*/metrics.csv and plots them.
   - Easy toggles at top to include/exclude prefixes or specific columns.
2) For each experiment, creates TWO final bar plots from final_test_metrics.csv:
   - Utility (F1s + Accuracy + ROC-AUC): Alpha vs Beta vs (optional) CTGAN/CTAB/etc if present
   - Fairness gaps (DP / EO / EOd): Alpha vs Beta (and optional baselines if present)

Run from project root:
  python plot_metrics.py
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("training_runs")

# ============================================================
# 0) QUICK TOGGLES (comment out what you don't want)
# ============================================================
PLOT_TIME_SERIES = True
PLOT_FINAL_BARS_UTILITY = True
PLOT_FINAL_BARS_FAIRNESS = True
SHOW_CURRICULUM_LINES = True

SMOOTH_WINDOW = 25  # set to 1 or None to disable smoothing

# --- easiest way to control which time-series get plotted ---
# Option A (recommended): choose prefixes (everything starting with these)
TIME_SERIES_INCLUDE_PREFIXES = [
    "meta", "global", "utility", "fairness", "local", "extra", "align"
]
# Example:
# TIME_SERIES_INCLUDE_PREFIXES = ["meta", "global", "utility"]

# Option B: additionally blacklist specific columns (even if prefix is included)
TIME_SERIES_EXCLUDE_COLS = {
    "wall_seconds",
    "align.reward_mode",  # usually string
}

# If you prefer plotting ONLY a few specific columns, set this to a non-empty list:
# (when non-empty, it overrides prefix logic)
TIME_SERIES_ONLY_COLS = [
    "meta.avg_reward",
    "global.global_obj",
    "global.local_reward",
]

# Column name for curriculum stage (vertical lines)
STAGE_COL = "align.curriculum_stage"


EXPERIMENT_LABELS = {

    # ── v3 Census (6000 eps, single-phase) ──────────────────────────────────
    "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_97f273d4": "v3 Census: Global+Anchors",
    "SPECv3_census_ablation_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_2f9e695b": "v3 Census: Global+Full",
    "SPECv3_census_ablation_global_hard_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_64d2ac98": "v3 Census: Global+Hard",
    "SPECv3_census_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_a14e2643": "v3 Census: Global Only",

    # ── v11 Two-Phase DVRL (1000 eps / phase) ───────────────────────────────
    "SPECv11_census_dvrl_twophase_1000_EP1000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603161455_5919adff": "v11 Census: DVRL TwoPhase",
    "SPECv11_credit_dvrl_twophase_1000_EP1000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603161455_00f93975": "v11 Credit: DVRL TwoPhase",
#     "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071844_7fee08dc": "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071844_7fee08dc",
#     "SPECv3_credit_ablation_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071927_0f291fd9": "SPECv3_credit_ablation_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071927_0f291fd9",
#     "SPECv3_credit_ablation_global_hard_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071924_a805a833": "SPECv3_credit_ablation_global_hard_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071924_a805a833",
#     "SPECv3_credit_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071958_80b395b6": "SPECv3_credit_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071958_80b395b6",
#     "SPECv3_credit_budget_high_ratio_EP6000_PCA10_REWfairness_minID1_majID0_TRJ6000_REAL3000_GG202603071838_8d4e5a4a": "SPECv3_credit_budget_high_ratio_EP6000_PCA10_REWfairness_minID1_majID0_TRJ6000_REAL3000_GG202603071838_8d4e5a4a",
#     "SPECv3_credit_budget_more_synth_EP6000_PCA10_REWfairness_minID1_majID0_TRJ4000_REAL3000_GG202603071841_5661416d": "SPECv3_credit_budget_more_synth_EP6000_PCA10_REWfairness_minID1_majID0_TRJ4000_REAL3000_GG202603071841_5661416d",
#     "SPECv3_credit_budget_scale_up_EP6000_PCA10_REWfairness_minID1_majID0_TRJ4000_REAL6000_GG202603071838_7453c23a": "SPECv3_credit_budget_scale_up_EP6000_PCA10_REWfairness_minID1_majID0_TRJ4000_REAL6000_GG202603071838_7453c23a",
#     "SPECv3_credit_budget_scale_up_large_EP6000_PCA10_REWfairness_minID1_majID0_TRJ6000_REAL9000_GG202603072007_c8eb463d": "SPECv3_credit_budget_scale_up_large_EP6000_PCA10_REWfairness_minID1_majID0_TRJ6000_REAL9000_GG202603072007_c8eb463d",
#     "SPECv3_credit_budget_very_high_ratio_EP6000_PCA10_REWfairness_minID1_majID0_TRJ9000_REAL3000_GG202603072007_ac3cf5b3": "SPECv3_credit_budget_very_high_ratio_EP6000_PCA10_REWfairness_minID1_majID0_TRJ9000_REAL3000_GG202603072007_ac3cf5b3",
 }

# Paired baselines: BASELINE_LABELS[i] is plotted alongside EXPERIMENT_LABELS[i]
# in the same final bar chart (baseline beta shown as an extra bar group).
BASELINE_LABELS = {
#     "BASELINE_folder_name": "Baseline Label",
}
# ---------------------------------------------------------------------
# Helpers to identify experiment + seed folders
# ---------------------------------------------------------------------
def is_seed_dir(p: Path) -> bool:
    return (
        p.is_dir()
        and p.name.startswith("seed_")
        and (p / "metrics.csv").is_file()
        and (p / "meta.json").is_file()
    )

def is_experiment_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    seeds = [d for d in p.iterdir() if is_seed_dir(d)]
    return len(seeds) > 0

# ---------------------------------------------------------------------
# Parse experiment name (for auto labels if needed)
# ---------------------------------------------------------------------
CORE_RE = re.compile(
    r"^EP(?P<episodes>\d+)_"
    r"PCA(?P<pca>\d+)_"
    r"REW(?P<rew>.+?)_"
    r"minID(?P<min>\d+)_"
    r"majID(?P<maj>\d+)_"
    r"TRJ(?P<trj>\d+)_"
    r"REAL(?P<real>\d+)_"
    r"BIAS(?P<bias>[0-9.]+)"
)

def split_core_and_tail(name: str):
    if "_DATE" in name:
        core, tail = name.split("_DATE", 1)
        return core, tail
    return name, ""

def parse_experiment_name(name: str):
    core, _ = split_core_and_tail(name)
    m = CORE_RE.match(core)
    if not m:
        return {"core": core}
    d = m.groupdict()
    d["core"] = core
    for k in ("episodes", "pca", "min", "maj", "trj", "real"):
        d[k] = int(d[k])
    d["bias"] = float(d["bias"])
    return d

def make_auto_label(info: dict) -> str:
    if not info:
        return "unknown"
    return (
        f"PCA{info.get('pca', '?')}_"
        f"min{info.get('min', '?')}_"
        f"maj{info.get('maj', '?')}_"
        f"bias{info.get('bias', '?')}"
    )

def label_for_folder(folder_name: str, info: dict) -> str:
    if folder_name in EXPERIMENT_LABELS:
        return EXPERIMENT_LABELS[folder_name]
    return make_auto_label(info)

# ---------------------------------------------------------------------
# Training metrics (metrics.csv)
# ---------------------------------------------------------------------
def load_seed_metrics(seed_dir: Path) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(seed_dir / "metrics.csv")
    except Exception as e:
        print(f"[warn] Could not read {seed_dir / 'metrics.csv'}: {e}")
        return None

    if "episode" not in df.columns:
        print(f"[warn] No 'episode' column in {seed_dir / 'metrics.csv'}, skipping this seed.")
        return None

    return df.copy()

def load_experiment_metrics_all_seeds(exp_dir: Path) -> pd.DataFrame | None:
    seed_dirs = [d for d in exp_dir.iterdir() if is_seed_dir(d)]
    if not seed_dirs:
        print(f"[info] No seeds found in {exp_dir}, skipping training curves.")
        return None

    dfs = []
    for sd in seed_dirs:
        df = load_seed_metrics(sd)
        if df is not None and not df.empty:
            df["seed"] = sd.name
            dfs.append(df)

    if not dfs:
        print(f"[info] All seeds in {exp_dir} unreadable/empty, skipping training curves.")
        return None

    all_df = pd.concat(dfs, ignore_index=True)
    all_df = all_df.sort_values(["episode", "seed"]).reset_index(drop=True)

    # Two-phase mode: make episode numbers continuous across phases.
    # Phase 1 episodes are [1..N]; Phase 2 resets to [1..N] — offset phase 2 by N.
    if "meta.phase" in all_df.columns:
        phase_vals = all_df["meta.phase"].fillna("")
        p1_mask = phase_vals.str.contains("phase1", na=False)
        p2_mask = phase_vals.str.contains("phase2", na=False)
        if p1_mask.any() and p2_mask.any():
            p1_max = all_df.loc[p1_mask, "episode"].max()
            all_df.loc[p2_mask, "episode"] = all_df.loc[p2_mask, "episode"] + p1_max
            all_df.attrs["phase2_start_ep"] = int(p1_max) + 1

    return all_df

# ---------------------------------------------------------------------
# Final test metrics (final_test_metrics.csv)
# ---------------------------------------------------------------------
def aggregate_final_metrics_for_experiment(exp_dir: Path) -> pd.Series | None:
    path = exp_dir / "final_test_metrics.csv"
    if not path.is_file():
        print(f"[info] No final_test_metrics.csv found in {exp_dir.name}, skipping final bars.")
        return None

    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"[warn] Could not read {path}: {e}")
        return None

    if df.empty:
        print(f"[info] final_test_metrics.csv is empty for {exp_dir.name}.")
        return None

    return df.mean(numeric_only=True)

def smooth_agg_df(agg: pd.DataFrame, window: int | None) -> pd.DataFrame:
    if window is None or window <= 1:
        return agg
    agg = agg.sort_values("episode").reset_index(drop=True)
    for col in ("mean", "min", "max"):
        if col in agg.columns:
            agg[col] = agg[col].rolling(window=window, center=True, min_periods=1).mean()
    return agg

# ============================================================
# Time-series discovery + labeling
# ============================================================
def _prefix_of(col: str) -> str:
    return col.split(".", 1)[0] if "." in col else col

def discover_time_series_metrics(seed_cache: dict[str, pd.DataFrame]) -> list[str]:
    """
    Union all numeric metric columns across all experiments (excluding episode/seed/stage),
    then filter using TIME_SERIES_ONLY_COLS or prefix include/exclude lists.
    """
    all_cols = set()
    for _exp, df in seed_cache.items():
        for c in df.columns:
            if c in ("episode", "seed", STAGE_COL):
                continue
            if c in TIME_SERIES_EXCLUDE_COLS:
                continue
            if pd.api.types.is_numeric_dtype(df[c]):
                all_cols.add(c)

    # override: explicit list
    if TIME_SERIES_ONLY_COLS:
        wanted = []
        for c in TIME_SERIES_ONLY_COLS:
            if c in all_cols:
                wanted.append(c)
            else:
                print(f"[warn] TIME_SERIES_ONLY_COLS requested '{c}' but it wasn't found as numeric anywhere.")
        return wanted

    cols = sorted(all_cols)

    if TIME_SERIES_INCLUDE_PREFIXES:
        inc = set(TIME_SERIES_INCLUDE_PREFIXES)
        cols = [c for c in cols if _prefix_of(c) in inc]

    cols = sorted(cols, key=lambda c: (_prefix_of(c), c))
    return cols

def pretty_metric_label(col: str) -> str:
    return col.replace(".", " · ").replace("_", " ")

# ============================================================
# FINAL BAR PLOTS
# ============================================================
def _annotate(ax_, rects):
    for r in rects:
        h = r.get_height()
        if np.isfinite(h):
            ax_.annotate(
                f"{h:.3f}",
                xy=(r.get_x() + r.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8
            )

def _discover_prefixes_for_utility(mean_series: pd.Series) -> list[str]:
    """
    Finds prefixes that have the standard utility suffixes in final_test_metrics.csv:
      <prefix>_f1_minority, _f1_majority, _f1_weighted, _acc, _roc_auc
    e.g. alpha, beta, alphaCT, alphaCTAB, oversample, undersample, jitter, ...
    """
    prefixes = set()
    for col in mean_series.index:
        if not isinstance(col, str):
            continue
        m = re.match(r"^(?P<prefix>.+?)_(f1_minority|f1_majority|f1_weighted|acc|roc_auc)$", col)
        if m:
            prefixes.add(m.group("prefix"))

    preferred = ["alpha", "beta", "alphaCT", "alphaCTAB", "oversample", "undersample", "jitter"]
    ordered = [p for p in preferred if p in prefixes] + sorted([p for p in prefixes if p not in preferred])
    return ordered

def plot_final_utility_bars(exp_label: str, mean_series: pd.Series):
    categories = [
        ("f1_minority", "F1 Minority"),
        ("f1_majority", "F1 Majority"),
        ("f1_weighted", "F1 Weighted"),
        ("acc",         "Accuracy"),
        ("roc_auc",     "ROC-AUC"),
    ]

    prefixes = _discover_prefixes_for_utility(mean_series)
    if not prefixes:
        print(f"[info] No utility prefix metrics found for {exp_label}. Skipping utility bars.")
        return None

    vals = []
    for p in prefixes:
        row = []
        for suf, _lbl in categories:
            key = f"{p}_{suf}"
            v = mean_series.get(key, np.nan)
            row.append(float(v) if v is not None else np.nan)
        vals.append(row)
    vals = np.array(vals, dtype=float)

    if np.all(np.isnan(vals)):
        print(f"[info] Utility bars all-NaN for {exp_label}. Skipping.")
        return None

    x = np.arange(len(categories))
    n = len(prefixes)
    group_width = 0.80
    bar_w = group_width / max(n, 1)
    offsets = (np.arange(n) - (n - 1) / 2.0) * bar_w

    fig, ax = plt.subplots(figsize=(11.5, 5))
    for i, p in enumerate(prefixes):
        bars = ax.bar(x + offsets[i], vals[i], bar_w, label=p)
        _annotate(ax, bars)

    ax.set_ylabel("Score (higher is better)")
    ax.set_title(f"Final Test Utility — All Benchmarks\n{exp_label}")
    ax.set_xticks(x)
    ax.set_xticklabels([lbl for _s, lbl in categories], rotation=30, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(fontsize="small")
    fig.tight_layout()
    return fig

def _discover_prefixes_for_fairness(mean_series: pd.Series) -> list[str]:
    """
    Finds prefixes that have fairness columns in final_test_metrics.csv.
    Required at least dp_diff or eo_tpr_diff (we'll plot what exists).
    Typically alpha_*, beta_*.
    """
    prefixes = set()
    for col in mean_series.index:
        if not isinstance(col, str) or "_" not in col:
            continue
        # match alpha_dp_diff, beta_eod_avg_diff, etc.
        m = re.match(r"^(?P<prefix>.+?)_(dp_diff|eo_tpr_diff|eod_max_diff|eod_avg_diff)$", col)
        if m:
            prefixes.add(m.group("prefix"))

    preferred = ["alpha", "beta", "alphaCT", "alphaCTAB", "oversample", "undersample", "jitter"]
    ordered = [p for p in preferred if p in prefixes] + sorted([p for p in prefixes if p not in preferred])
    return ordered

def plot_final_fairness_bars(exp_label: str, mean_series: pd.Series):
    fairness_metrics = [
        ("dp_diff",      "DP |ΔP(ŷ=1)|"),
        ("eo_tpr_diff",  "EO |ΔTPR|"),
        ("eod_max_diff", "EOd max(|ΔTPR|,|ΔFPR|)"),
        ("eod_avg_diff", "EOd avg(|ΔTPR|,|ΔFPR|)"),
    ]

    prefixes = _discover_prefixes_for_fairness(mean_series)
    if not prefixes:
        print(f"[info] No fairness columns found for {exp_label}. Skipping fairness bars.")
        return None

    # Build values: rows=prefixes, cols=fairness_metrics
    vals = []
    for p in prefixes:
        row = []
        for suf, _lbl in fairness_metrics:
            row.append(float(mean_series.get(f"{p}_{suf}", np.nan)))
        vals.append(row)
    vals = np.array(vals, dtype=float)

    if np.all(np.isnan(vals)):
        print(f"[info] Fairness bars all-NaN for {exp_label}. Skipping.")
        return None

    x = np.arange(len(fairness_metrics))
    n = len(prefixes)
    group_width = 0.80
    bar_w = group_width / max(n, 1)
    offsets = (np.arange(n) - (n - 1) / 2.0) * bar_w

    fig, ax = plt.subplots(figsize=(11.5, 4.9))
    for i, p in enumerate(prefixes):
        bars = ax.bar(x + offsets[i], vals[i], bar_w, label=p)
        _annotate(ax, bars)

    # scale y so tiny gaps are visible
    finite = vals[np.isfinite(vals)]
    ymax = 0.1 if finite.size == 0 else float(max(0.05, np.max(finite) * 1.25))

    ax.set_ylim(0.0, ymax)
    ax.set_ylabel("Gap (lower is better)")
    ax.set_title(f"Final Test Fairness Gaps\n{exp_label}")
    ax.set_xticks(x)
    ax.set_xticklabels([lbl for _s, lbl in fairness_metrics], rotation=25, ha="right")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(fontsize="small")
    fig.tight_layout()
    return fig

# ---------------------------------------------------------------------
# Main plotting logic
# ---------------------------------------------------------------------
def main():
    if not ROOT.is_dir():
        print(f"[error] {ROOT} does not exist. Run this from the project root.")
        return

    # --- Decide which experiment folders to use ---
    if EXPERIMENT_LABELS:
        exp_dirs = []
        for folder_name in EXPERIMENT_LABELS.keys():
            p = ROOT / folder_name
            if not is_experiment_dir(p):
                print(f"[warn] {p} is not a valid experiment folder (missing seeds/metrics). Skipping.")
                continue
            exp_dirs.append(p)
        exp_dirs = sorted(exp_dirs, key=lambda p: p.name)
        print(f"[info] Using {len(exp_dirs)} experiment folders from EXPERIMENT_LABELS.")
    else:
        exp_dirs = [d for d in ROOT.iterdir() if is_experiment_dir(d)]
        exp_dirs = sorted(exp_dirs, key=lambda p: p.name)
        print(f"[info] Auto-discovered {len(exp_dirs)} experiment folders under {ROOT}.")

    if not exp_dirs:
        print("[info] No experiment folders selected/found. Nothing to plot.")
        return

    seed_cache: dict[str, pd.DataFrame] = {}
    info_cache: dict[str, dict] = {}
    final_cache: dict[str, pd.Series] = {}
    phase_transitions: dict[str, int] = {}  # exp_name -> first episode of phase 2

    for exp_dir in exp_dirs:
        name = exp_dir.name
        info_cache[name] = parse_experiment_name(name)

        all_df = load_experiment_metrics_all_seeds(exp_dir)
        if all_df is not None:
            seed_cache[name] = all_df
            if "phase2_start_ep" in all_df.attrs:
                phase_transitions[name] = all_df.attrs["phase2_start_ep"]

        final_series = aggregate_final_metrics_for_experiment(exp_dir)
        if final_series is not None:
            final_cache[name] = final_series

    # ============================================================
    # Precompute curriculum stage transition episodes (across all runs)
    # ============================================================
    stage_transitions = []
    if SHOW_CURRICULUM_LINES:
        stage_set = set()
        for _name, df in seed_cache.items():
            if STAGE_COL not in df.columns:
                continue
            stage_by_ep = (
                df.groupby("episode")[STAGE_COL]
                .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
                .sort_index()
            )
            episodes = stage_by_ep.index.to_numpy()
            stages = stage_by_ep.to_numpy()
            for i in range(1, len(stages)):
                if stages[i] != stages[i - 1]:
                    stage_set.add(float(episodes[i]))
        stage_transitions = sorted(stage_set)

    # ============================================================
    # 1) Time-series plots (AUTO: discover all numeric metrics)
    # ============================================================
    if PLOT_TIME_SERIES:
        metric_cols = discover_time_series_metrics(seed_cache)
        print(f"[info] Discovered {len(metric_cols)} numeric time-series metrics to plot.")

        for metric_col in metric_cols:
            plt.figure()
            plotted_any = False
            metric_label = pretty_metric_label(metric_col)

            for exp_dir in exp_dirs:
                name = exp_dir.name
                if name not in seed_cache:
                    continue

                df = seed_cache[name]
                if metric_col not in df.columns:
                    continue

                # aggregate across seeds per episode
                agg = (
                    df.groupby("episode")[metric_col]
                    .agg(["mean", "min", "max"])
                    .reset_index()
                    .sort_values("episode")
                )
                agg = smooth_agg_df(agg, SMOOTH_WINDOW)

                info = info_cache.get(name, {})
                label = label_for_folder(name, info)

                line = plt.plot(agg["episode"], agg["mean"], label=label, linewidth=2.0, zorder=3)[0]
                color = line.get_color()

                plt.fill_between(
                    agg["episode"],
                    agg["min"],
                    agg["max"],
                    color=color,
                    alpha=0.30,
                    linewidth=0,
                    zorder=1,
                )
                plt.plot(agg["episode"], agg["min"], color=color, alpha=0.35, linewidth=0.8, zorder=2)
                plt.plot(agg["episode"], agg["max"], color=color, alpha=0.35, linewidth=0.8, zorder=2)

                plotted_any = True

            if not plotted_any:
                plt.close()
                continue

            if SHOW_CURRICULUM_LINES:
                for ep in stage_transitions:
                    plt.axvline(ep, linestyle="--", color="gray", alpha=0.3)

            # Phase boundary: one vertical line per two-phase experiment plotted here.
            # Collect unique phase-transition episodes across experiments shown in this figure.
            phase_eps_in_fig = sorted(set(
                phase_transitions[name]
                for name in (d.name for d in exp_dirs)
                if name in seed_cache
                and name in phase_transitions
                and metric_col in seed_cache[name].columns
            ))
            for p_ep in phase_eps_in_fig:
                plt.axvline(p_ep, linestyle="-", color="#cc0000", alpha=0.7, linewidth=1.8, zorder=5)
                ymin, ymax = plt.ylim()
                plt.text(
                    p_ep + 0.5, ymax * 0.97,
                    "Phase 2
(y=0 gen.)",
                    color="#cc0000", fontsize=8, va="top", ha="left",
                    bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2),
                    zorder=6,
                )

            plt.xlabel("Episode")
            plt.ylabel(metric_label)
            plt.title(metric_label)
            plt.grid(True)
            plt.legend(fontsize="small")
            plt.tight_layout()

    # ============================================================
    # 2) Final bar plots: Utility + Fairness (TWO figures per run)
    # ============================================================
    if (PLOT_FINAL_BARS_UTILITY or PLOT_FINAL_BARS_FAIRNESS) and not final_cache:
        print("[info] No final_test_metrics.csv found in selected experiments; skipping final bars.")

    # Build index-aligned baseline lookup
    exp_keys  = list(EXPERIMENT_LABELS.keys())
    base_keys = list(BASELINE_LABELS.keys())

    for exp_dir in exp_dirs:
        name = exp_dir.name
        if name not in final_cache:
            continue

        mean_series = final_cache[name].copy()
        info        = info_cache.get(name, {})
        exp_label   = label_for_folder(name, info)

        # Inject paired baseline beta metrics (if any) as extra prefix columns
        if name in exp_keys:
            idx = exp_keys.index(name)
            if idx < len(base_keys):
                base_folder = base_keys[idx]
                base_label  = BASELINE_LABELS[base_folder]
                base_series = aggregate_final_metrics_for_experiment(ROOT / base_folder)
                if base_series is not None:
                    # Prefix = label with spaces replaced by underscores
                    pfx = base_label.replace(" ", "_")
                    for col, val in base_series.items():
                        if col.startswith("beta_"):
                            mean_series[f"{pfx}_{col[len('beta_'):]}"] = val

        if PLOT_FINAL_BARS_UTILITY:
            plot_final_utility_bars(exp_label, mean_series)

        if PLOT_FINAL_BARS_FAIRNESS:
            plot_final_fairness_bars(exp_label, mean_series)


    plt.show()

if __name__ == "__main__":
    main()


In [4]:
#1312
# | Metric                  | Average       |
# | ----------------------- | ------------- |
# | **alphaCT_f1_minority** | **0.7855041** |
# | **alphaCT_f1_majority** | **0.6153448** |
# | **alphaCT_f1_weighted** | **0.6952365** |
# | **alphaCT_f1_macro**    | **0.7004245** |
# | **alphaCT_brier**       | **0.1979598** |

# #134
# | Metric                  | Average       |
# | ----------------------- | ------------- |
# | **alphaCT_f1_minority** | **0.9015152** |
# | **alphaCT_f1_majority** | **0.9417683** |
# | **alphaCT_f1_weighted** | **0.9283892** |
# | **alphaCT_f1_macro**    | **0.9216417** |
# | **alphaCT_brier**       | **0.0575585** |

#10
# | Metric                  | Average       |
# | ----------------------- | ------------- |
# | **alphaCT_f1_minority** | **0.6445781** |
# | **alphaCT_f1_majority** | **0.8482108** |
# | **alphaCT_f1_weighted** | **0.7991551** |
# | **alphaCT_f1_macro**    | **0.7463944** |
# | **alphaCT_brier**       | **0.1438272** |



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re

ROOT = Path("training_runs")

# ============================================================
# 1) SELECT WHICH EXPERIMENTS TO PLOT
# ============================================================
EXPERIMENT_LABELS = {
    "EP6000_PCA10_REWlocal_gauss_minID13_majID12_TRJ2000_REAL3000_BIAS0.35_G202512151601_75165367": "PCA10_min13_maj12_bias0.35",
    "EP6000_PCA10_REWlocal_gauss_minID13_majID4_TRJ2000_REAL3000_BIAS0.35_G202512151601_d941e1bb": "PCA10_min13_maj4_bias0.35",
    "EP6000_PCA10_REWlocal_gauss_minID1_majID0_TRJ2000_REAL3000_BIAS0.35_G202512151601_01804334": "PCA10_min1_maj0_bias0.35",
}

# ============================================================
# 2) MANUAL CTGAN BASELINES (keyed by (minID, majID))
#    Values are your alphaCT_* averages
# ============================================================
CTGAN_BASELINES = {
    (13, 12): {
        "f1_minority": 0.7855041,
        "f1_majority": 0.6153448,
        "f1_weighted": 0.6952365,
        "acc":         np.nan,
        "roc_auc":     np.nan,
    },
    (13, 4): {
        "f1_minority": 0.9015152,
        "f1_majority": 0.9417683,
        "f1_weighted": 0.9283892,
        "acc":         np.nan,
        "roc_auc":     np.nan,
    },
    (1, 0): {
        "f1_minority": 0.6445781,
        "f1_majority": 0.8482108,
        "f1_weighted": 0.7991551,
        "acc":         np.nan,
        "roc_auc":     np.nan,
    },
}

# ============================================================
# Parse minID / majID from experiment folder name
# ============================================================
MINMAJ_RE = re.compile(r"minID(?P<min>\d+)_majID(?P<maj>\d+)")

def parse_min_maj(exp_name: str) -> tuple[int | None, int | None]:
    m = MINMAJ_RE.search(exp_name)
    if not m:
        return None, None
    return int(m.group("min")), int(m.group("maj"))

# ============================================================
# Load final metrics
# ============================================================
def aggregate_final_metrics_for_experiment(exp_dir: Path) -> pd.Series | None:
    path = exp_dir / "final_test_metrics.csv"
    if not path.is_file():
        print(f"[info] No final_test_metrics.csv found in {exp_dir.name}, skipping.")
        return None
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"[warn] Could not read {path}: {e}")
        return None
    if df.empty:
        print(f"[info] final_test_metrics.csv is empty for {exp_dir.name}.")
        return None
    return df.mean(numeric_only=True)

# ============================================================
# Plot: Alpha vs Beta vs CTGAN (utility) + Fairness bar plot
# ============================================================
def plot_alpha_beta_ctgan(exp_label: str, mean_series: pd.Series, ctgan: dict | None):
    # -----------------------------
    # 1) Utility plot
    # -----------------------------
    categories = ["F1 Minority", "F1 Majority", "F1 Weighted", "Accuracy", "ROC-AUC"]
    x = np.arange(len(categories))

    alpha_vals = np.array([
        mean_series.get("alpha_f1_minority", np.nan),
        mean_series.get("alpha_f1_majority", np.nan),
        mean_series.get("alpha_f1_weighted", np.nan),
        mean_series.get("alpha_acc", np.nan),
        mean_series.get("alpha_roc_auc", np.nan),
    ], dtype=float)

    beta_vals = np.array([
        mean_series.get("beta_f1_minority", np.nan),
        mean_series.get("beta_f1_majority", np.nan),
        mean_series.get("beta_f1_weighted", np.nan),
        mean_series.get("beta_acc", np.nan),
        mean_series.get("beta_roc_auc", np.nan),
    ], dtype=float)

    if ctgan is None:
        ctgan_vals = np.array([np.nan] * len(categories), dtype=float)
    else:
        ctgan_vals = np.array([
            ctgan.get("f1_minority", np.nan),
            ctgan.get("f1_majority", np.nan),
            ctgan.get("f1_weighted", np.nan),
            ctgan.get("acc", np.nan),
            ctgan.get("roc_auc", np.nan),
        ], dtype=float)

    if np.all(np.isnan(alpha_vals)) and np.all(np.isnan(beta_vals)) and np.all(np.isnan(ctgan_vals)):
        print(f"[info] No usable utility metrics for {exp_label}, skipping.")
        return

    width = 0.25
    fig, ax = plt.subplots(figsize=(11, 5))

    r1 = ax.bar(x - width, alpha_vals, width, label="Alpha")
    r2 = ax.bar(x,         beta_vals,  width, label="Beta")
    r3 = ax.bar(x + width, ctgan_vals, width, label="CTGAN")

    def annotate(ax_, rects):
        for r in rects:
            h = r.get_height()
            if not np.isnan(h):
                ax_.annotate(
                    f"{h:.3f}",
                    xy=(r.get_x() + r.get_width()/2, h),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=8
                )

    annotate(ax, r1); annotate(ax, r2); annotate(ax, r3)

    # Optional: show % change vs CTGAN (Beta relative to CTGAN)
    for i in range(len(categories)):
        c = ctgan_vals[i]
        b = beta_vals[i]
        if np.isnan(c) or np.isnan(b) or c == 0:
            continue
        pct = (b - c) / c * 100.0
        ax.text(
            x[i],
            0.05,
            f"Beta vs CTGAN: {pct:+.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none")
        )

    ax.set_ylabel("Score")
    ax.set_title(f"Final Test Utility — Alpha vs Beta vs CTGAN\n{exp_label}")
    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=30, ha="right")
    ax.set_ylim(0.0, 1.05)
    ax.legend()
    plt.tight_layout()

    # -----------------------------
    # 2) Fairness plot
    # -----------------------------
    fairness_metrics = [
        ("dp_diff",      "DP |ΔP(ŷ=1)|"),
        ("eo_tpr_diff",  "EO |ΔTPR|"),
        ("eod_max_diff", "EOd max(|ΔTPR|,|ΔFPR|)"),
    ]

    f_cats = [lbl for _, lbl in fairness_metrics]
    xf = np.arange(len(f_cats))

    alpha_fair = np.array([mean_series.get(f"alpha_{k}", np.nan) for k, _ in fairness_metrics], dtype=float)
    beta_fair  = np.array([mean_series.get(f"beta_{k}",  np.nan) for k, _ in fairness_metrics], dtype=float)

    if ctgan is None:
        ctgan_fair = np.array([np.nan] * len(fairness_metrics), dtype=float)
    else:
        ctgan_fair = np.array([ctgan.get(k, np.nan) for k, _ in fairness_metrics], dtype=float)

    if np.all(np.isnan(alpha_fair)) and np.all(np.isnan(beta_fair)) and np.all(np.isnan(ctgan_fair)):
        print(f"[info] No usable fairness metrics for {exp_label}, skipping fairness plot.")
        return

    fig2, ax2 = plt.subplots(figsize=(11, 4.6))

    rf1 = ax2.bar(xf - width, alpha_fair, width, label="Alpha")
    rf2 = ax2.bar(xf,         beta_fair,  width, label="Beta")
    rf3 = ax2.bar(xf + width, ctgan_fair, width, label="CTGAN")

    annotate(ax2, rf1); annotate(ax2, rf2); annotate(ax2, rf3)

    finite = np.array([v for v in np.concatenate([alpha_fair, beta_fair, ctgan_fair]) if np.isfinite(v)], dtype=float)
    if finite.size == 0:
        ymax = 1.0
    else:
        ymax = float(np.max(finite))
        ymax = max(0.1, min(1.5, ymax * 1.15))

    ax2.set_ylabel("Gap (lower is better)")
    ax2.set_title(f"Final Test Fairness Gaps — Alpha vs Beta vs CTGAN\n{exp_label}")
    ax2.set_xticks(xf)
    ax2.set_xticklabels(f_cats, rotation=25, ha="right")
    ax2.set_ylim(0.0, ymax)
    ax2.grid(True, axis="y", alpha=0.25)
    ax2.legend()
    plt.tight_layout()


def main():
    if not ROOT.is_dir():
        print(f"[error] {ROOT} does not exist. Run this from the project root.")
        return

    for folder_name, exp_label in EXPERIMENT_LABELS.items():
        exp_dir = ROOT / folder_name
        mean_series = aggregate_final_metrics_for_experiment(exp_dir)
        if mean_series is None:
            continue

        min_id, maj_id = parse_min_maj(folder_name)
        ctgan = None
        if min_id is not None and maj_id is not None:
            ctgan = CTGAN_BASELINES.get((min_id, maj_id), None)
            if ctgan is None:
                print(f"[warn] No CTGAN baseline for (minID={min_id}, majID={maj_id}) -> {exp_label}")

        plot_alpha_beta_ctgan(exp_label, mean_series, ctgan)

    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

ROOT = Path("training_runs")

# ============================================================
# 1) SELECT WHICH EXPERIMENTS TO PLOT
# ============================================================

DATASETS = {
    "census_income": {
        "v3_runs": {
            "SPECv3_census_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_a14e2643": "RL v3: Global Only",
            "SPECv3_census_ablation_global_hard_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_64d2ac98": "RL v3: Global+Hard",
            "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_97f273d4": "RL v3: Global+Anchors",
            "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603121842_acafe162": "RL v3: Global+Anchors (2)",
            "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603131643_0520d348": "RL v3: Global+Anchors (3)",
            "SPECv3_census_ablation_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_2f9e695b": "RL v3: Full",
        },
        "v5_spec_prefix": "v5_census_uncertainty_anchors",
        "v5_label": "RL v5: UA",
        "new_runs": {
            "SPECv7_census_hard_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603141507_7ee3460c": "RL v7: Hard Anchors",
            "SPECv9_census_sigma1_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603151451_1118f0e9": "RL v9: Sigma1",
            "SPECv9_census_sigma1_highlr_neverreset_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603152014_05015a96": "RL v9: Sigma1+HiLR+NoReset",
            "SPECv11_census_dvrl_twophase_1000_EP1000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603161455_5919adff": "RL v11: DVRL TwoPhase",
        },
        "baselines": {
            "BASELINE_gaussian_ot_repair_baseline_otrep_census_seed42_c861f183__G202603161804": "OT Repair",
            "BASELINE_group_dro_baseline_gdro_census_seed42_7d9629d9__G202603161801": "Group DRO",
        },
    },
    "credit_card": {
        "v3_runs": {
            "SPECv3_credit_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071958_80b395b6": "RL v3: Global Only",
            "SPECv3_credit_ablation_global_hard_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071924_a805a833": "RL v3: Global+Hard",
            "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071844_7fee08dc": "RL v3: Global+Anchors",
            "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603121915_d2289bb2": "RL v3: Global+Anchors (2)",
            "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603131745_7772f394": "RL v3: Global+Anchors (3)",
            "SPECv3_credit_ablation_global_full_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071927_0f291fd9": "RL v3: Full",
        },
        "v5_spec_prefix": "v5_credit_uncertainty_anchors",
        "v5_label": "RL v5: UA",
        "new_runs": {
            "SPECv7_credit_hard_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603141513_6748a1da": "RL v7: Hard Anchors",
            "SPECv8_credit_whiten_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603141802_30a45f02": "RL v8: Whiten",
            "SPECv9_credit_sigma2_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603151451_1bce1c3e": "RL v9: Sigma2",
            "SPECv9_credit_sigma2_neverreset_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603152014_02112548": "RL v9: Sigma2+NoReset",
            "SPECv11_credit_dvrl_twophase_1000_EP1000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603161455_00f93975": "RL v11: DVRL TwoPhase",
        },
        "baselines": {
            "BASELINE_gaussian_ot_repair_baseline_otrep_credit_6882b6a2__G202603111900": "OT Repair",
            "BASELINE_group_dro_baseline_gdro_credit_153d34a0__G202603111900": "Group DRO",
            "BASELINE_group_dro_baseline_gdro_credit_0f79e8fe__G202603121736": "Group DRO (2)",
        },
    },
}

# Metrics to display for RL/baseline methods (beta_* columns)
FAIRNESS_METRICS = [
    ("beta_eo_tpr_diff",  "EO |ΔTPR|"),
    ("beta_dp_diff",      "DP gap"),
    ("beta_eod_max_diff", "EOd max"),
]
UTILITY_METRICS = [
    ("beta_f1_minority",  "F1 Minority"),
    ("beta_f1_weighted",  "F1 Weighted"),
    ("beta_roc_auc",      "ROC-AUC"),
]

# Metrics to display for the no-augmentation alpha reference (alpha_* columns)
ALPHA_METRICS_F = [
    ("alpha_eo_tpr_diff",  "EO |ΔTPR|"),
    ("alpha_dp_diff",      "DP gap"),
    ("alpha_eod_max_diff", "EOd max"),
]
ALPHA_METRICS_U = [
    ("alpha_f1_minority",  "F1 Minority"),
    ("alpha_f1_weighted",  "F1 Weighted"),
    ("alpha_roc_auc",      "ROC-AUC"),
]

ALL_TABLE_COLS = [
    ("eo_tpr_diff",  "EO"),
    ("dp_diff",      "DP"),
    ("eod_max_diff", "EOd"),
    ("f1_minority",  "F1 Minority"),
    ("f1_weighted",  "F1 Weighted"),
    ("roc_auc",      "ROC-AUC"),
]


def is_experiment_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    return any(
        d.is_dir() and d.name.startswith("seed_") and (d / "metrics.csv").is_file()
        for d in p.iterdir()
    )


def find_runs_by_spec_prefix(spec_prefix: str, label: str) -> dict:
    """Auto-discover training_run dirs whose SPEC name starts with spec_prefix."""
    results = {}
    for d in sorted(ROOT.glob(f"SPEC{spec_prefix}_*")):
        if is_experiment_dir(d):
            results[d.name] = label
    return results


def load_stats(folder: str) -> tuple[pd.Series, pd.Series]:
    """Return (mean, std) across seeds from final_test_metrics.csv."""
    path = ROOT / folder / "final_test_metrics.csv"
    df = pd.read_csv(path)
    return df.mean(numeric_only=True), df.std(numeric_only=True)


def fmt(mean, std):
    if np.isnan(mean):
        return "—"
    if std is None or np.isnan(std):
        return f"{mean:.4f}"
    return f"{mean:.4f} ± {std:.4f}"


# ============================================================
# Build data per dataset
# ============================================================

DATASET_TITLES = {"census_income": "Census Income", "credit_card": "Credit Card"}

fig, axes = plt.subplots(
    nrows=2, ncols=2,
    figsize=(16, 11),
    constrained_layout=True,
)
fig.suptitle("Final Test Results: RL v3 Ablations + v5 UA vs Baselines", fontsize=14, fontweight="bold")

RL_COLORS    = ["#4C72B0", "#6699CC", "#2F5597", "#88AADD", "#B0C4DE", "#A8C4E0"]
V5_COLORS    = ["#2ca02c", "#98df8a", "#17becf"]
BASE_COLORS  = ["#DD8452", "#55A868", "#C44E52", "#8172B2"]

for row_idx, (dataset_key, cfg) in enumerate(DATASETS.items()):
    methods = {}  # label -> (mean, std)

    # Combine v3 + v5 RL runs
    v5_runs = find_runs_by_spec_prefix(cfg["v5_spec_prefix"], cfg["v5_label"])
    new_runs = cfg.get("new_runs", {})
    all_rl_runs = {**cfg["v3_runs"], **v5_runs, **new_runs}

    if v5_runs:
        print(f"[info] {dataset_key}: found {len(v5_runs)} v5 run(s)")
    if new_runs:
        print(f"[info] {dataset_key}: found {len(new_runs)} new run(s): {list(new_runs.values())}")

    # No-augmentation reference: average alpha_* across all v3 runs for this dataset
    alpha_means = []
    for folder in cfg["v3_runs"]:
        try:
            m, s = load_stats(folder)
            alpha_means.append(m)
        except Exception as e:
            print(f"[warn] {folder}: {e}")
    if alpha_means:
        alpha_mean = pd.concat(alpha_means, axis=1).mean(axis=1)
        methods["No Aug. (Alpha)"] = (alpha_mean, None)

    for folder, label in all_rl_runs.items():
        try:
            m, s = load_stats(folder)
            methods[label] = (m, s)
        except Exception as e:
            print(f"[warn] {folder}: {e}")

    for folder, label in cfg["baselines"].items():
        try:
            m, s = load_stats(folder)
            methods[label] = (m, s)
        except Exception as e:
            print(f"[warn] {folder}: {e}")

    labels = list(methods.keys())
    n = len(labels)

    rl_count = 0
    v5_count = 0
    base_count = 0
    colors = []
    for lbl in labels:
        if lbl == "No Aug. (Alpha)":
            colors.append("#888888")
        elif lbl.startswith("RL v5:"):
            colors.append(V5_COLORS[v5_count % len(V5_COLORS)])
            v5_count += 1
        elif lbl.startswith("RL v11:"):
            colors.append("#d62728")  # bold red for v11 DVRL TwoPhase
        elif lbl.startswith("RL v7:") or lbl.startswith("RL v8:") or lbl.startswith("RL v9:"):
            NEW_COLORS = ["#9467bd", "#e377c2", "#bcbd22", "#17becf"]
            colors.append(NEW_COLORS[rl_count % len(NEW_COLORS)])
            rl_count += 1
        elif lbl.startswith("RL:") or lbl.startswith("RL v3:"):
            colors.append(RL_COLORS[rl_count % len(RL_COLORS)])
            rl_count += 1
        else:
            colors.append(BASE_COLORS[base_count % len(BASE_COLORS)])
            base_count += 1

    x = np.arange(len(FAIRNESS_METRICS))
    bar_w = 0.80 / max(n, 1)
    offsets = (np.arange(n) - (n - 1) / 2.0) * bar_w

    # ---------- Fairness subplot ----------
    ax_f = axes[row_idx, 0]
    for i, (lbl, (m, s)) in enumerate(methods.items()):
        metric_list_f = ALPHA_METRICS_F if lbl == "No Aug. (Alpha)" else FAIRNESS_METRICS
        vals = np.array([float(m.get(col, np.nan)) for col, _ in metric_list_f])
        errs = np.array([float(s.get(col, np.nan)) for col, _ in metric_list_f]) if s is not None else np.zeros_like(vals)
        errs = np.where(np.isnan(errs), 0.0, errs)
        ax_f.bar(x + offsets[i], vals, bar_w, color=colors[i], label=lbl,
                 yerr=errs, capsize=2, error_kw={"elinewidth": 0.8})

    ax_f.set_title(f"{DATASET_TITLES[dataset_key]} — Fairness (↓ better)")
    ax_f.set_xticks(x)
    ax_f.set_xticklabels([lbl for _, lbl in FAIRNESS_METRICS])
    ax_f.set_ylabel("Gap")
    ax_f.grid(True, axis="y", alpha=0.3)
    all_vals_f = [float(m.get(col, np.nan))
                  for lbl, (m, _) in methods.items()
                  for col, _ in (ALPHA_METRICS_F if lbl == "No Aug. (Alpha)" else FAIRNESS_METRICS)]
    ymax_f = max(0.05, np.nanmax(all_vals_f) * 1.2) if any(np.isfinite(all_vals_f)) else 0.5
    ax_f.set_ylim(0, ymax_f)
    ax_f.legend(fontsize=7, loc="upper right")

    # ---------- Utility subplot ----------
    ax_u = axes[row_idx, 1]
    x_u = np.arange(len(UTILITY_METRICS))
    for i, (lbl, (m, s)) in enumerate(methods.items()):
        metric_list_u = ALPHA_METRICS_U if lbl == "No Aug. (Alpha)" else UTILITY_METRICS
        vals = np.array([float(m.get(col, np.nan)) for col, _ in metric_list_u])
        errs = np.array([float(s.get(col, np.nan)) for col, _ in metric_list_u]) if s is not None else np.zeros_like(vals)
        errs = np.where(np.isnan(errs), 0.0, errs)
        ax_u.bar(x_u + offsets[i], vals, bar_w, color=colors[i], label=lbl,
                 yerr=errs, capsize=2, error_kw={"elinewidth": 0.8})

    ax_u.set_title(f"{DATASET_TITLES[dataset_key]} — Utility (↑ better)")
    ax_u.set_xticks(x_u)
    ax_u.set_xticklabels([lbl for _, lbl in UTILITY_METRICS])
    ax_u.set_ylabel("Score")
    ax_u.set_ylim(0, 1.05)
    ax_u.grid(True, axis="y", alpha=0.3)
    ax_u.legend(fontsize=7, loc="lower right")

    # ---------- Results table ----------
    col_names = [disp for _, disp in ALL_TABLE_COLS]
    rows = {}
    for lbl, (m, s) in methods.items():
        is_alpha = (lbl == "No Aug. (Alpha)")
        row = []
        for col_key, _ in ALL_TABLE_COLS:
            prefix = "alpha" if is_alpha else "beta"
            mean_val = float(m.get(f"{prefix}_{col_key}", np.nan))
            std_val  = float(s.get(f"{prefix}_{col_key}", np.nan)) if s is not None else float("nan")
            row.append(fmt(mean_val, std_val))
        rows[lbl] = row

    table_df = pd.DataFrame(rows, index=col_names).T
    table_df.index.name = "Method"
    print(f"\n{DATASET_TITLES[dataset_key]}")
    display(table_df.style.set_caption(DATASET_TITLES[dataset_key]))

plt.show()


In [ ]:

# ============================================================
# FAIRNESS-UTILITY TRADEOFF ANALYSIS
# ============================================================
# Two Pareto scatter plots per dataset (EO vs F1_min, EO vs AUC)
# + comprehensive tradeoff table.
#
# Key metrics:
#   Δ EO (%)         — relative EO reduction vs alpha (positive = better fairness)
#   Δ F1_min (pp)    — absolute F1_minority change vs alpha in percentage points
#   Δ F1_w (pp)      — absolute F1_weighted change
#   Δ AUC (pp)       — absolute ROC-AUC change
#   Δ ACC (pp)       — absolute accuracy change
#   UAFI             — Utility-Adjusted Fairness Index (multi-metric)
#                      harmonic mean of fairness_gain and composite utility retention
#                      fairness_gain       = max(0, (EO_alpha - EO) / EO_alpha)
#                      utility_retention   = mean([F1w, AUC, ACC] / alpha) clipped to 1
#                      Range [0,1]; higher = more fairness improvement per unit utility cost
#   Util. Preserved  — ✓ if ALL of {F1_w, AUC, ACC} stay within −0.5pp of alpha
#   Win              — ✓ if EO improves AND Util. Preserved (the key publishable claim)
#   Pareto (EO/F1m)  — ✓ if not dominated on (EO↓, F1_minority↑)
#   Pareto (EO/AUC)  — ✓ if not dominated on (EO↓, AUC↑)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from pathlib import Path
from IPython.display import display

ROOT = Path("training_runs")
UTIL_TOL = 0.005  # 0.5pp tolerance for "utility preserved"

def is_experiment_dir(p):
    if not p.is_dir():
        return False
    return any(
        d.is_dir() and d.name.startswith("seed_") and (d / "metrics.csv").is_file()
        for d in p.iterdir()
    )

def find_runs_by_spec_prefix(spec_prefix, label):
    results = {}
    for d in sorted(ROOT.glob(f"SPEC{spec_prefix}_*")):
        if is_experiment_dir(d):
            results[d.name] = label
    return results

def load_stats(folder):
    path = ROOT / folder / "final_test_metrics.csv"
    df = pd.read_csv(path)
    return df.mean(numeric_only=True), df.std(numeric_only=True)

def _get(series, key):
    if series is None:
        return np.nan
    val = series.get(key, np.nan)
    return float(val) if (val == val) else np.nan

def uafi_multi(eo, eo_a, f1w, f1w_a, auc, auc_a, acc, acc_a):
    """Multi-metric UAFI: harmonic mean of fairness_gain and composite utility retention."""
    fg = max(0.0, (eo_a - eo) / (eo_a + 1e-9))
    retentions = []
    for v, va in [(f1w, f1w_a), (auc, auc_a), (acc, acc_a)]:
        if not (np.isnan(v) or np.isnan(va) or va < 1e-9):
            retentions.append(min(1.0, v / va))
    if not retentions:
        return np.nan
    ur = np.mean(retentions)
    if fg + ur < 1e-9:
        return 0.0
    return 2.0 * fg * ur / (fg + ur)

def pareto_mask(x_vals, y_vals):
    """True if point is NOT dominated (x↓ better, y↑ better)."""
    n = len(x_vals)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if x_vals[j] <= x_vals[i] and y_vals[j] >= y_vals[i]:
                if x_vals[j] < x_vals[i] or y_vals[j] > y_vals[i]:
                    dominated[i] = True
                    break
    return ~dominated

DATASETS_T = {
    "census_income": {
        "v3_runs": {
            "SPECv3_census_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_a14e2643":     "RL: Global Only",
            "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_97f273d4":  "RL: v3 Anchors (1)",
            "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603121842_acafe162":  "RL: v3 Anchors (2)",
            "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603131643_0520d348":  "RL: v3 Anchors (3)",
        },
        "extra_prefixes": [
            ("v5_census_uncertainty_anchors",  "RL: v5 UA"),
            ("v7_census_hard_anchors_EP6000",  "RL: v7 Hard Anchors"),   # full runs only
        ],
        "baselines": {
            "BASELINE_gaussian_ot_repair_baseline_otrep_census_a8b46764__G202603111900": "OT Repair",
            "BASELINE_group_dro_baseline_gdro_census_738fc989__G202603111900":           "Group DRO",
            "BASELINE_group_dro_baseline_gdro_census_663e8736__G202603121735":           "Group DRO (2)",
        },
    },
    "credit_card": {
        "v3_runs": {
            "SPECv3_credit_ablation_global_only_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071958_80b395b6":    "RL: Global Only",
            "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071844_7fee08dc": "RL: v3 Anchors (1)",
            "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603121915_d2289bb2": "RL: v3 Anchors (2)",
            "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603131745_7772f394": "RL: v3 Anchors (3)",
        },
        "extra_prefixes": [
            ("v5_credit_uncertainty_anchors",  "RL: v5 UA"),
            ("v7_credit_hard_anchors_EP6000",  "RL: v7 Hard Anchors"),   # full runs only
        ],
        "baselines": {
            "BASELINE_gaussian_ot_repair_baseline_otrep_credit_6882b6a2__G202603111900": "OT Repair",
            "BASELINE_group_dro_baseline_gdro_credit_153d34a0__G202603111900":           "Group DRO",
            "BASELINE_group_dro_baseline_gdro_credit_0f79e8fe__G202603121736":           "Group DRO (2)",
        },
    },
}

METHOD_COLORS = {
    "No Aug. (Alpha)":      "#888888",
    "RL: Global Only":      "#6699CC",
    "RL: v3 Anchors (1)":  "#4C72B0",
    "RL: v3 Anchors (2)":  "#2F5597",
    "RL: v3 Anchors (3)":  "#1A3A7A",
    "RL: v5 UA":            "#2ca02c",
    "RL: v7 Hard Anchors": "#d62728",
    "RL v11: DVRL TwoPhase": "#ff7f0e",
    "OT Repair":            "#DD8452",
    "Group DRO":            "#55A868",
    "Group DRO (2)":        "#8172B2",
}
DATASET_TITLES = {"census_income": "Census Income", "credit_card": "Credit Card"}

def _scatter_panel(ax, plot_data, pareto_flags, x_col, y_col, xlabel, ylabel, title):
    """Generic scatter + Pareto frontier for any (x↓, y↑) pair."""
    labels_p = list(plot_data.keys())
    x_arr = np.array([plot_data[l][x_col] for l in labels_p])
    y_arr = np.array([plot_data[l][y_col] for l in labels_p])
    is_p  = pareto_flags[f"{x_col}_{y_col}"]

    for i, label in enumerate(labels_p):
        x, y = x_arr[i], y_arr[i]
        if np.isnan(x) or np.isnan(y):
            continue
        color  = METHOD_COLORS.get(label, "#333333")
        marker = "*" if is_p[i] else "o"
        ms     = 200 if is_p[i] else 80
        ax.scatter(x, y, c=color, s=ms, marker=marker, zorder=3,
                   edgecolors="white" if is_p[i] else color, linewidths=0.8)
        ax.annotate(label, (x, y), fontsize=6.2, color=color,
                    xytext=(5, 3), textcoords="offset points",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    # Pareto step line
    pareto_pts = sorted(
        [(x_arr[i], y_arr[i]) for i in range(len(labels_p))
         if is_p[i] and not (np.isnan(x_arr[i]) or np.isnan(y_arr[i]))],
        key=lambda t: t[0])
    if len(pareto_pts) >= 2:
        px, py = zip(*pareto_pts)
        ax.step(px, py, where="post", color="#aaaaaa", lw=1.2, ls="--", alpha=0.6, zorder=1)

    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.22)


# ── Main loop ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11), constrained_layout=True)
fig.suptitle("Fairness–Utility Pareto Frontiers  (★ = Pareto-optimal)", fontsize=13, fontweight="bold")

all_tables = {}

for row_idx, (dataset_key, cfg) in enumerate(DATASETS_T.items()):

    # ── Load methods ─────────────────────────────────────────────────────
    methods = {}
    for folder, label in {**cfg["v3_runs"], **cfg["baselines"]}.items():
        try:
            m, s = load_stats(folder)
            methods[label] = (m, s)
        except Exception as e:
            print(f"[warn] {folder}: {e}")

    for prefix, label in cfg["extra_prefixes"]:
        for folder, lbl in find_runs_by_spec_prefix(prefix, label).items():
            try:
                m, s = load_stats(folder)
                if lbl in methods:
                    old_m, _ = methods[lbl]
                    methods[lbl] = (pd.concat([old_m, m], axis=1).mean(axis=1), None)
                else:
                    methods[lbl] = (m, s)
            except Exception as e:
                print(f"[warn] {folder}: {e}")

    # ── Alpha reference (mean of RL runs) ────────────────────────────────
    rl_means = [m for lbl, (m, _) in methods.items() if lbl.startswith("RL:")]
    alpha_ref = pd.concat(rl_means, axis=1).mean(axis=1) if rl_means else None

    def ga(key):   return _get(alpha_ref, f"alpha_{key}")
    eo_a  = ga("eo_tpr_diff"); f1w_a = ga("f1_weighted")
    auc_a = ga("roc_auc");     acc_a = ga("acc")
    f1m_a = ga("f1_minority")

    methods = {"No Aug. (Alpha)": (alpha_ref, None), **methods}

    # ── Build per-method numeric dict ────────────────────────────────────
    plot_data = {}
    for label, (m, s) in methods.items():
        is_a = (label == "No Aug. (Alpha)")
        px = "alpha" if is_a else "beta"
        def gv(key): return _get(m, f"{px}_{key}")
        eo  = gv("eo_tpr_diff"); f1m = gv("f1_minority")
        f1w = gv("f1_weighted"); auc = gv("roc_auc"); acc = gv("acc")
        if not np.isnan(eo):
            plot_data[label] = {"eo": eo, "f1m": f1m, "f1w": f1w, "auc": auc, "acc": acc,
                                "std_eo":  _get(s, f"{px}_eo_tpr_diff") if s is not None else np.nan,
                                "std_f1m": _get(s, f"{px}_f1_minority") if s is not None else np.nan,
                                "std_auc": _get(s, f"{px}_roc_auc")     if s is not None else np.nan,
                                "std_acc": _get(s, f"{px}_acc")         if s is not None else np.nan}

    labels_p = list(plot_data.keys())
    eo_arr  = np.array([plot_data[l]["eo"]  for l in labels_p])
    f1m_arr = np.array([plot_data[l]["f1m"] for l in labels_p])
    auc_arr = np.array([plot_data[l]["auc"] for l in labels_p])
    acc_arr = np.array([plot_data[l]["acc"] for l in labels_p])

    pareto_flags = {
        "eo_f1m": pareto_mask(eo_arr, f1m_arr),
        "eo_auc": pareto_mask(eo_arr, auc_arr),
    }

    # ── Scatter panels ────────────────────────────────────────────────────
    _scatter_panel(axes[row_idx, 0], plot_data, pareto_flags,
                   "eo", "f1m", "EO |ΔTPR| (↓)", "F1 Minority (↑)",
                   f"{DATASET_TITLES[dataset_key]}  ·  EO vs F1 Minority")

    _scatter_panel(axes[row_idx, 1], plot_data, pareto_flags,
                   "eo", "auc", "EO |ΔTPR| (↓)", "ROC-AUC (↑)",
                   f"{DATASET_TITLES[dataset_key]}  ·  EO vs ROC-AUC")

    # ── Tradeoff table ────────────────────────────────────────────────────
    rows = []
    for label, d in plot_data.items():
        eo  = d["eo"];  f1m = d["f1m"]
        f1w = d["f1w"]; auc = d["auc"]; acc = d["acc"]

        def pct(v, va):
            return (va - v) / (va + 1e-9) * 100 if not (np.isnan(v) or np.isnan(va)) else np.nan
        def pp(v, va):
            return (v - va) * 100 if not (np.isnan(v) or np.isnan(va)) else np.nan

        d_eo  = pct(eo,  eo_a)
        d_f1m = pp(f1m, f1m_a);  d_f1w = pp(f1w, f1w_a)
        d_auc = pp(auc, auc_a);  d_acc = pp(acc, acc_a)
        score = uafi_multi(eo, eo_a, f1w, f1w_a, auc, auc_a, acc, acc_a)

        # Utility preserved: all 3 metrics within -UTIL_TOL of alpha
        util_ok = all(
            (not np.isnan(delta)) and (delta >= -UTIL_TOL * 100)
            for delta in [d_f1w, d_auc, d_acc]
        )
        eo_improved = (not np.isnan(d_eo)) and d_eo > 0
        win = eo_improved and util_ok

        # Pareto memberships
        pf1m = "✓" if (label in labels_p and pareto_flags["eo_f1m"][labels_p.index(label)]) else ""
        pauc = "✓" if (label in labels_p and pareto_flags["eo_auc"][labels_p.index(label)]) else ""

        def fmt_delta(v, positive_good=True):
            if np.isnan(v): return "—"
            sign = "+" if v >= 0 else ""
            color_hint = ""  # plain text; styling done via pandas Styler below
            return f"{sign}{v:.2f}"

        rows.append({
            "Method":        label,
            "EO":            f"{eo:.4f}"  if not np.isnan(eo)  else "—",
            "Δ EO %":        fmt_delta(d_eo),
            "F1 Min":        f"{f1m:.4f}" if not np.isnan(f1m) else "—",
            "Δ F1m pp":      fmt_delta(d_f1m),
            "Δ F1w pp":      fmt_delta(d_f1w),
            "Δ AUC pp":      fmt_delta(d_auc),
            "Δ ACC pp":      fmt_delta(d_acc),
            "UAFI":          f"{score:.3f}" if not np.isnan(score) else "—",
            "Util. Pres.":   "✓" if util_ok else "",
            "Win":           "✓" if win    else "",
            "Pareto F1m":    pf1m,
            "Pareto AUC":    pauc,
        })

    table_df = pd.DataFrame(rows).set_index("Method")
    all_tables[dataset_key] = table_df

plt.show()

# ── Print tables ─────────────────────────────────────────────────────────────
print("\n━━━━  FAIRNESS–UTILITY TRADEOFF SUMMARY  ━━━━")
print(f"UAFI   = harmonic mean of fairness_gain & composite utility retention (F1w + AUC + ACC)")
print(f"Win    = EO improves  AND  all utility metrics within −{UTIL_TOL*100:.1f}pp of alpha")
print(f"Pareto = not dominated on that (EO↓, metric↑) pair\n")

def highlight(row):
    """Green background for Win rows, yellow for Util. Pres. only."""
    styles = [""] * len(row)
    if row.get("Win") == "✓":
        return ["background-color: #d4edda"] * len(row)
    if row.get("Util. Pres.") == "✓":
        return ["background-color: #fff3cd"] * len(row)
    return styles

for dataset_key, tbl in all_tables.items():
    print(f"{'─'*70}")
    print(f"  {DATASET_TITLES[dataset_key]}")
    print(f"{'─'*70}")
    display(
        tbl.style
           .apply(highlight, axis=1)
           .set_caption(
               f"{DATASET_TITLES[dataset_key]} — green = fairness improved + utility fully preserved; "
               f"yellow = utility preserved only"
           )
    )
    print()
